# 260504 view history delete

`260430_membership_v1`의 `_v1.csv` 파일을 소스로, `View_History_v1.csv`에 등장하지 않는 `USER_NUM`을 가진 유저를 제거한다.

- `View_History_v1.csv`에 없는 `USER_NUM`을 `User_Mapping_v1.csv`에서 제거한다.
- `User_Mapping_v1.csv`에서 제거된 `USER_KEY`에 해당하는 행을 `Membership_v1.csv`에서도 제거한다.
- `Movie_Master_v1.csv`는 원본 그대로 유지한다.
- 결과는 `_v2.csv`로 저장한다.

In [9]:
from pathlib import Path

import pandas as pd


SOURCE_DIR = Path.cwd().parent / "260430_membership_v1(이상치, 이름변경)"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

membership = pd.read_csv(SOURCE_DIR / "Membership_v1.csv")
user_mapping = pd.read_csv(SOURCE_DIR / "User_Mapping_v1.csv")
view_history = pd.read_csv(SOURCE_DIR / "View_History_v1.csv")
movie_master = pd.read_csv(SOURCE_DIR / "Movie_Master_v1.csv")

print("membership:   ", membership.shape)
print("user_mapping: ", user_mapping.shape)
print("view_history: ", view_history.shape)
print("movie_master: ", movie_master.shape)

membership:    (17876, 15)
user_mapping:  (19877, 2)
view_history:  (106205, 5)
movie_master:  (14018, 3)


In [10]:
# View_History에 존재하는 USER_NUM 집합
user_nums_with_history = set(view_history["USER_NUM"].dropna().unique())

# User_Mapping 전체 USER_NUM 집합
user_nums_all = set(user_mapping["USER_NUM"].dropna().unique())

# View_History 없는 USER_NUM
user_nums_no_history = user_nums_all - user_nums_with_history

print(f"전체 USER_NUM 수:              {len(user_nums_all):,}")
print(f"View_History 있는 USER_NUM 수: {len(user_nums_with_history):,}")
print(f"View_History 없는 USER_NUM 수: {len(user_nums_no_history):,}  → 삭제 대상")

전체 USER_NUM 수:              19,877
View_History 있는 USER_NUM 수: 14,892
View_History 없는 USER_NUM 수: 4,985  → 삭제 대상


In [11]:
# User_Mapping 필터링: View_History에 없는 USER_NUM 제거
user_mapping_clean = user_mapping.loc[
    user_mapping["USER_NUM"].isin(user_nums_with_history)
].copy()

# 남은 USER_KEY 집합
remaining_user_keys = set(user_mapping_clean["USER_KEY"].dropna().unique())

# Membership 필터링: 남은 USER_KEY만 유지
membership_clean = membership.loc[
    membership["USER_KEY"].isin(remaining_user_keys)
].copy()

# View_History: 이미 기준이므로 그대로 사용
view_history_clean = view_history.copy()

# Movie_Master: 그대로 유지
movie_master_clean = movie_master.copy()

summary = pd.DataFrame([
    {"data": "membership",   "before": len(membership),   "after": len(membership_clean),   "removed": len(membership)   - len(membership_clean)},
    {"data": "user_mapping", "before": len(user_mapping), "after": len(user_mapping_clean), "removed": len(user_mapping) - len(user_mapping_clean)},
    {"data": "view_history", "before": len(view_history), "after": len(view_history_clean), "removed": len(view_history) - len(view_history_clean)},
    {"data": "movie_master", "before": len(movie_master), "after": len(movie_master_clean), "removed": len(movie_master) - len(movie_master_clean)},
])

print("=" * 52)
print("[ 삭제 결과 ]")
print("=" * 52)
print(f"  삭제된 USER_NUM 수: {len(user_nums_no_history):,}개")
print("-" * 52)
for _, row in summary.iterrows():
    print(f"  {row['data']:15s}  {row['before']:>6,}행 → {row['after']:>6,}행  (삭제: {row['removed']:>5,}행)")
print("=" * 52)

summary

[ 삭제 결과 ]
  삭제된 USER_NUM 수: 4,985개
----------------------------------------------------
  membership       17,876행 → 14,873행  (삭제: 3,003행)
  user_mapping     19,877행 → 14,892행  (삭제: 4,985행)
  view_history     106,205행 → 106,205행  (삭제:     0행)
  movie_master     14,018행 → 14,018행  (삭제:     0행)


,data,before,after,removed
0,membership,17876,14873,3003
1,user_mapping,19877,14892,4985
2,view_history,106205,106205,0
3,movie_master,14018,14018,0


In [12]:
output_map = {
    "Membership_v2.csv":   membership_clean,
    "User_Mapping_v2.csv": user_mapping_clean,
    "View_History_v2.csv": view_history_clean,
    "Movie_Master_v2.csv": movie_master_clean,
}

for file_name, data in output_map.items():
    output_path = OUTPUT_DIR / file_name
    data.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"saved: {output_path.name}  rows={len(data):,}")

saved: Membership_v2.csv  rows=14,873
saved: User_Mapping_v2.csv  rows=14,892
saved: View_History_v2.csv  rows=106,205
saved: Movie_Master_v2.csv  rows=14,018
